In [ ]:
#| default_exp cache

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import time, pickle, json, base64, threading, functools, math, random
import numpy as np
from pathlib import Path
from fastcore.all import store_attr, AttrDict, L, ifnone
from apswutils.db import NotFoundError
from litesearch.core import database

__all__ = ['Cache', 'SemanticCache', 'FanoutCache']


## Internals

In [ ]:
#| export
_DEFAULT_MODEL = 'minishlab/potion-base-8M'
_MISSING = object()
_KEY_LOCKS: dict = {}
_KEY_LOCKS_MUTEX = threading.Lock()

def _key_lock(k: str) -> threading.Lock:
    with _KEY_LOCKS_MUTEX:
        if k not in _KEY_LOCKS: _KEY_LOCKS[k] = threading.Lock()
        return _KEY_LOCKS[k]

def _make_cache_key(func, args, kwargs, typed: bool) -> str:
    parts = [f'{func.__module__}.{func.__qualname__}', repr(args), repr(sorted(kwargs.items()))]
    if typed: parts += [repr(type(a)) for a in args]
    return ':'.join(parts)


In [ ]:
#| export
class _CacheTransaction:
    def __init__(self, db): self._db = db
    def __enter__(self): self._db.execute('BEGIN EXCLUSIVE'); return self
    def __exit__(self, exc, *_): self._db.execute('ROLLBACK' if exc else 'COMMIT'); return False


## Cache

In [ ]:
#| export
class Cache:
    'diskcache-compatible SQLite key-value cache with TTL, LRU eviction, and stampede protection.'
    def __init__(self,
                 directory='~/.litesearch/cache',  # path for the SQLite DB (or :memory:)
                 size_limit=2**30,                 # max bytes before LRU eviction
                 ttl=None,                         # default TTL in seconds (None = no expiry)
                 table='cache',                    # table name inside the DB
    ):
        store_attr()
        db_path = str(Path(directory).expanduser()) if directory != ':memory:' else ':memory:'
        self.db = database(db_path, sem_search=False)
        self._tbl = self.db.t[table].create(
            key=str, raw=bytes, expire=float, tag=str,
            store_time=float, access_time=float, hits=int, size=int, compute_time=float,
            pk='key',
            defaults={'hits': 0},
            if_not_exists=True)
        self._tbl.create_index(['access_time'], if_not_exists=True)
        self._tbl.create_index(['tag'], if_not_exists=True)
        self._meta = self.db.t['_cache_meta'].create(
            key=str, val=int, pk='key', defaults={'val': 0}, if_not_exists=True)
        self._meta.insert_all([{'key': 'hits', 'val': 0}, {'key': 'misses', 'val': 0}], replace=True)

    def set(self, key, value, expire=None, tag=None):
        'Store a value; expire is seconds from now (overrides instance ttl).'
        now = time.time()
        exp = now + expire if expire else (now + self.ttl if self.ttl else None)
        raw = pickle.dumps(value, protocol=pickle.HIGHEST_PROTOCOL)
        self._tbl.upsert({'key': str(key), 'raw': raw, 'expire': exp, 'tag': tag,
                          'store_time': now, 'access_time': now, 'hits': 0,
                          'size': len(raw), 'compute_time': None})
        self._evict_to_size_limit()
        return True

    def get(self, key, default=None):
        'Return cached value, or default on miss/expiry.'
        try: row = self._tbl.get(str(key))
        except NotFoundError:
            self._meta.update({'val': self._meta.get('misses')['val'] + 1}, pk_values='misses')
            return default
        if row['expire'] and row['expire'] < time.time():
            self._tbl.delete(str(key))
            self._meta.update({'val': self._meta.get('misses')['val'] + 1}, pk_values='misses')
            return default
        self._tbl.update({'access_time': time.time(), 'hits': row['hits'] + 1}, pk_values=str(key))
        self._meta.update({'val': self._meta.get('hits')['val'] + 1}, pk_values='hits')
        return pickle.loads(row['raw'])

    def delete(self, key):
        'Delete a key; returns True if it existed.'
        try: self._tbl.delete(str(key)); return True
        except: return False

    def __setitem__(self, k, v): self.set(k, v)
    def __getitem__(self, k):
        v = self.get(k, _MISSING)
        if v is _MISSING: raise KeyError(k)
        return v
    def __delitem__(self, k): self.delete(k)
    def __contains__(self, key):
        return bool(list(self._tbl.rows_where(
            'key=? AND (expire IS NULL OR expire>?)', [str(key), time.time()])))
    def __len__(self):
        return self.db.execute(f'SELECT COUNT(*) FROM "{self._tbl.name}"').fetchone()[0]
    def __iter__(self):
        for row in self._tbl.rows_where('expire IS NULL OR expire>?', [time.time()], select='key'):
            yield row['key']

    def expire(self):
        'Delete all expired entries; returns count removed.'
        before = len(self)
        self._tbl.delete_where('expire IS NOT NULL AND expire<?', [time.time()])
        return before - len(self)

    def evict(self, tag):
        'Delete all entries with the given tag; returns count removed.'
        before = len(self)
        self._tbl.delete_where('tag=?', [tag])
        return before - len(self)

    def clear(self):
        'Delete all entries; returns count removed.'
        n = len(self)
        self._tbl.delete_where()
        self._meta.insert_all([{'key': 'hits', 'val': 0}, {'key': 'misses', 'val': 0}], replace=True)
        return n

    def volume(self):
        'Total bytes used by stored values.'
        return self.db.execute(f'SELECT COALESCE(SUM(size),0) FROM "{self._tbl.name}"').fetchone()[0]

    def stats(self, reset=False):
        'Return AttrDict(hits=, misses=). Pass reset=True to zero counters.'
        h = self._meta.get('hits')['val']
        m = self._meta.get('misses')['val']
        if reset:
            self._meta.insert_all([{'key': 'hits', 'val': 0}, {'key': 'misses', 'val': 0}], replace=True)
        return AttrDict(hits=h, misses=m)

    def _evict_to_size_limit(self):
        if self.volume() <= self.size_limit: return
        rows = list(self._tbl.rows_where(order_by='access_time'))
        freed, victims = 0, []
        for r in rows:
            if self.volume() - freed <= self.size_limit: break
            victims.append(r['key']); freed += r['size']
        if victims:
            self._tbl.delete_where(f"key IN ({','.join(['?']*len(victims))})", victims)

    def transact(self): return _CacheTransaction(self.db)
    def close(self): self.db.close()
    def __enter__(self): return self
    def __exit__(self, *_): self.close()

    def memoize(self, expire=None, typed=False, tag=None):
        'Decorator: cache results with double-checked locking (thundering herd safe).'
        def decorator(func):
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                k = _make_cache_key(func, args, kwargs, typed)
                result = self.get(k, _MISSING)
                if result is not _MISSING: return result
                with _key_lock(k):
                    result = self.get(k, _MISSING)
                    if result is not _MISSING: return result
                    result = func(*args, **kwargs)
                    self.set(k, result, expire=expire, tag=tag)
                return result
            return wrapper
        return decorator

    def memoize_stampede(self, expire=None, typed=False, tag=None, beta=1):
        'Decorator: XFetch probabilistic early recompute; keeps cache warm near TTL boundary.'
        def decorator(func):
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                k = _make_cache_key(func, args, kwargs, typed)
                try: row = self._tbl.get(k)
                except NotFoundError: row = None
                if row:
                    if row['expire']:
                        now = time.time()
                        delta = row['compute_time'] or 0.001
                        if row['expire'] - now > -delta * beta * math.log(random.random()):
                            return pickle.loads(row['raw'])
                    else:
                        return pickle.loads(row['raw'])
                with _key_lock(k):
                    t0 = time.time()
                    result = func(*args, **kwargs)
                    ct = time.time() - t0
                    raw = pickle.dumps(result, protocol=pickle.HIGHEST_PROTOCOL)
                    now = time.time()
                    exp_ts = now + expire if expire else (now + self.ttl if self.ttl else None)
                    self._tbl.upsert({'key': k, 'raw': raw, 'expire': exp_ts, 'tag': tag,
                                      'store_time': now, 'access_time': now, 'hits': 0,
                                      'size': len(raw), 'compute_time': ct})
                return result
            return wrapper
        return decorator


In [ ]:
show_doc(Cache)
show_doc(Cache.memoize)
show_doc(Cache.memoize_stampede)

In [ ]:
import time
c = Cache(':memory:')
c.set('k', 42)
assert c.get('k') == 42
assert 'k' in c
assert len(c) == 1
c.delete('k')
assert c.get('k') is None
c.close()


In [ ]:
c = Cache(':memory:')
c.set('x', 'val', expire=0.05)
assert c.get('x') == 'val'
time.sleep(0.1)
assert c.get('x') is None


In [ ]:
c = Cache(':memory:')
c.set('a', 1, tag='red'); c.set('b', 2, tag='red'); c.set('c', 3, tag='blue')
assert c.evict('red') == 2
assert len(c) == 1
assert c.clear() == 1
assert len(c) == 0


In [ ]:
c = Cache(':memory:')
c.set('k', 99); c.get('k'); c.get('missing')
s = c.stats()
assert s.hits == 1 and s.misses == 1
c.stats(reset=True)
assert c.stats().hits == 0


In [ ]:
with Cache(':memory:') as c:
    c['hello'] = 'world'
    assert c['hello'] == 'world'


In [ ]:
c = Cache(':memory:')
try:
    with c.transact():
        c.set('t', 1)
        raise ValueError
except ValueError:
    pass
assert c.get('t') is None


In [ ]:
c = Cache(':memory:')
call_count = 0
@c.memoize(expire=10)
def square(n): global call_count; call_count += 1; return n * n
assert square(4) == 16
assert square(4) == 16
assert call_count == 1


In [ ]:
import threading as _th
c = Cache(':memory:')
body_count = 0
@c.memoize()
def slow_fn():
    global body_count
    time.sleep(0.05)
    body_count += 1
    return 'done'
threads = [_th.Thread(target=slow_fn) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
assert body_count == 1, f'Expected 1 call, got {body_count}'


In [ ]:
c = Cache(':memory:')
@c.memoize(typed=True)
def identity(x): return x
identity(1); identity(1.0)
assert len(c) == 2  # int and float cached as separate keys


In [ ]:
c = Cache(':memory:')
@c.memoize(tag='math')
def cube(n): return n**3
cube(3)
assert c.evict('math') >= 1
assert c.stats(reset=True).hits == 0


In [ ]:
c = Cache(':memory:')
calls = 0
@c.memoize_stampede(expire=10, beta=1)
def fetch(x): global calls; calls += 1; return x * 3
assert fetch(4) == 12
assert fetch(4) == 12
assert calls == 1, f'Expected 1 compute, got {calls}'


In [ ]:
c = Cache(':memory:')
@c.memoize_stampede(expire=0.3, beta=99)
def slow_val(): time.sleep(0.01); return 'fresh'
slow_val()
time.sleep(0.1)
slow_val()
assert slow_val() == 'fresh'


In [ ]:
kv = Cache(':memory:')
def cache(p=None, ttl=3600, **_):
    def d(f): return kv.memoize_stampede(expire=ttl, tag=p)(f)
    return d
calls = 0
@cache(p='api', ttl=60)
def expensive_api_call(key): global calls; calls += 1; return f'result:{key}'
assert expensive_api_call('a') == 'result:a'
expensive_api_call('a')
assert calls == 1


## SemanticCache

In [ ]:
#| export
class SemanticCache:
    'Semantic key-value cache: paraphrase-tolerant lookups via FTS5 + vector RRF.'
    def __init__(self,
                 directory='~/.litesearch/semantic_cache',  # path for the SQLite DB (or :memory:)
                 encoder=None,    # FastEncode | model2vec StaticModel | any .encode()/.embed() | None → lazy model2vec
                 model=None,      # model ID string (used only when encoder=None)
                 ttl=None,        # default TTL in seconds
                 threshold=0.022, # min RRF score to accept a hit (0.030=exact, 0.022=para, 0.016=broad)
                 table='semantic_cache',
                 size_limit=2**30,
    ):
        store_attr()
        self._enc = None
        db_path = str(Path(directory).expanduser()) if directory != ':memory:' else ':memory:'
        self.db = database(db_path)
        self.store = self.db.get_store(name=table)
        self.db.execute(f'CREATE INDEX IF NOT EXISTS _idx_{table}_ts ON "{table}"(uploaded_at)')

    @property
    def enc(self):
        if self._enc: return self._enc
        if self.encoder: self._enc = self.encoder; return self._enc
        from model2vec import StaticModel
        self._enc = StaticModel.from_pretrained(ifnone(self.model, _DEFAULT_MODEL))
        return self._enc

    def _emb(self, key) -> bytes:
        enc = self.enc
        if   hasattr(enc, 'encode_query'): arr = enc.encode_query([key])[0]
        elif hasattr(enc, 'encode'):       arr = enc.encode([key])[0]
        elif hasattr(enc, 'embed'):        arr = enc.embed([key])[0]
        else: raise TypeError(f'{type(enc)} must have .encode() or .embed()')
        return np.asarray(arr).astype(np.float16).tobytes()

    def set(self, key, value, expire=None, tag=None):
        'Store a value under key (embedded for semantic lookup).'
        now = time.time()
        exp = now + expire if expire else (now + self.ttl if self.ttl else None)
        meta = {'_p': base64.b64encode(pickle.dumps(value)).decode(), 'expire': exp, 'tag': tag}
        self.store.insert({'content': str(key), 'embedding': self._emb(key),
                           'metadata': json.dumps(meta), 'uploaded_at': now})

    def get(self, key, default=None, threshold=None):
        'Semantic lookup; returns default if no hit above threshold.'
        results = self.db.search(q=str(key), emb=self._emb(key), table_name=self.table,
                                 columns=['metadata', 'uploaded_at'], limit=5)
        if not results: return default
        top = results[0]
        if top['_rrf_score'] < ifnone(threshold, self.threshold): return default
        if self.ttl and time.time() - top['uploaded_at'] > self.ttl: return default
        meta = json.loads(top['metadata'])
        if meta.get('expire') and time.time() > meta['expire']: return default
        return pickle.loads(base64.b64decode(meta['_p']))

    def purge(self):
        'Delete all entries; returns count.'
        n = self.db.execute(f'SELECT COUNT(*) FROM "{self.table}"').fetchone()[0]
        self.store.delete_where()
        return n

    def purge_expired(self, ttl=None):
        'Delete entries older than ttl seconds; returns count.'
        effective_ttl = ifnone(ttl, self.ttl)
        if not effective_ttl: return 0
        cutoff = time.time() - effective_ttl
        before = self.db.execute(f'SELECT COUNT(*) FROM "{self.table}"').fetchone()[0]
        self.store.delete_where('uploaded_at < ?', [cutoff])
        after = self.db.execute(f'SELECT COUNT(*) FROM "{self.table}"').fetchone()[0]
        return before - after

    def purge_semantic(self, q, threshold=None, limit=20, dry_run=False):
        'Delete entries semantically similar to q; returns L of victim rows.'
        thr = ifnone(threshold, self.threshold)
        results = self.db.search(q=q, emb=self._emb(q), table_name=self.table,
                                 columns=['metadata', 'uploaded_at'], limit=limit)
        victims = L(r for r in results if r['_rrf_score'] >= thr)
        if not dry_run:
            for v in victims: self.store.delete_where('rowid=?', [v['rowid']])
        return victims

    def purge_topic(self, q, threshold=None, include_expired=True, dry_run=False):
        'Purge expired + semantically similar entries; returns AttrDict(expired=, semantic=).'
        exp_count = self.purge_expired() if include_expired and not dry_run else 0
        sem = self.purge_semantic(q, threshold=threshold, dry_run=dry_run)
        return AttrDict(expired=exp_count, semantic=sem)

    def expire(self):
        'Delete entries whose per-record expire timestamp has passed; returns count.'
        tbl = self.table
        rows = self.db.execute(
            f'SELECT rowid FROM "{tbl}" WHERE json_extract(metadata,\'$.expire\') IS NOT NULL'
            f' AND json_extract(metadata,\'$.expire\') < {time.time()}').fetchall()
        for (rowid,) in rows: self.store.delete_where('rowid=?', [rowid])
        return len(rows)

    def evict(self, tag):
        'Delete all entries with the given tag; returns count.'
        tbl = self.table
        rows = self.db.execute(
            f'SELECT rowid FROM "{tbl}" WHERE json_extract(metadata,\'$.tag\')=?',
            [tag]).fetchall()
        for (rowid,) in rows: self.store.delete_where('rowid=?', [rowid])
        return len(rows)

    def stats(self):
        n = self.db.execute(f'SELECT COUNT(*) FROM "{self.table}"').fetchone()[0]
        return AttrDict(count=n, hits=0, misses=0)

    def transact(self): return _CacheTransaction(self.db)
    def close(self): self.db.close()
    def __enter__(self): return self
    def __exit__(self, *_): self.close()

    def memoize(self, expire=None, typed=False, tag=None):
        'Decorator: semantic-cache function results.'
        def decorator(func):
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                k = _make_cache_key(func, args, kwargs, typed)
                result = self.get(k, _MISSING)
                if result is not _MISSING: return result
                with _key_lock(k):
                    result = self.get(k, _MISSING)
                    if result is not _MISSING: return result
                    result = func(*args, **kwargs)
                    self.set(k, result, expire=expire, tag=tag)
                return result
            return wrapper
        return decorator


In [ ]:
show_doc(SemanticCache)
show_doc(SemanticCache.purge_semantic)

In [ ]:
#| eval: false
sc = SemanticCache(':memory:')
sc.set('capital of France', 'Paris')
assert sc.get('capital of France') == 'Paris'


In [ ]:
#| eval: false
sc = SemanticCache(':memory:')
sc.set('what is the capital of France', 'Paris')
result = sc.get('capital city of France')
assert result == 'Paris', f'Expected semantic hit, got {result!r}'


In [ ]:
#| eval: false
sc = SemanticCache(':memory:', threshold=0.99)
sc.set('what is the capital of France', 'Paris')
assert sc.get('population of Tokyo') is None


In [ ]:
#| eval: false
sc = SemanticCache(':memory:', ttl=0.05)
sc.set('x', 'y')
time.sleep(0.1)
assert sc.get('x') is None


In [ ]:
#| eval: false
sc = SemanticCache(':memory:', ttl=0.05)
sc.set('a', 1); sc.set('b', 2)
time.sleep(0.1)
n = sc.purge_expired()
assert n == 2


In [ ]:
#| eval: false
sc = SemanticCache(':memory:')
sc.set('machine learning basics', 'result')
victims = sc.purge_semantic('machine learning', dry_run=True)
assert len(victims) >= 1


In [ ]:
#| eval: false
sc = SemanticCache(':memory:')
calls = 0
@sc.memoize()
def answer(q): global calls; calls += 1; return f'answer to {q}'
answer('what is AI'); answer('what is AI')
assert calls == 1


In [ ]:
#| eval: false
sc = SemanticCache(':memory:')
sc.set('q1', 'v1', tag='batch1'); sc.set('q2', 'v2', tag='batch1')
sc.evict('batch1')
assert sc.get('q1') is None


In [ ]:
#| eval: false
from litesearch.utils import FastEncode, modernbert
enc = FastEncode(modernbert)
sc = SemanticCache(':memory:', encoder=enc)
sc.set('transformer architecture', 'attention is all you need')
assert sc.get('transformer architecture') == 'attention is all you need'


## FanoutCache

In [ ]:
#| export
class FanoutCache:
    'Hash-sharded cache; distributes keys across N Cache (or SemanticCache) instances.'
    def __init__(self,
                 directory='~/.litesearch/fanout',  # parent directory (or :memory:)
                 shards=8,                           # number of shards
                 cache_cls=None,                     # Cache (default) or SemanticCache
                 **kwargs,                           # forwarded to each shard constructor
    ):
        store_attr()
        cache_cls = cache_cls or Cache
        if directory != ':memory:':
            Path(directory).expanduser().mkdir(parents=True, exist_ok=True)
            shard_dirs = [str(Path(directory).expanduser() / f'shard_{i}') for i in range(shards)]
        else:
            shard_dirs = [':memory:'] * shards
        self._shards = [cache_cls(directory=d, **kwargs) for d in shard_dirs]

    def _shard(self, key): return self._shards[hash(str(key)) % self.shards]

    def set(self, key, value, expire=None, tag=None):
        return self._shard(key).set(key, value, expire=expire, tag=tag)
    def get(self, key, default=None): return self._shard(key).get(key, default=default)
    def delete(self, key): return self._shard(key).delete(key)
    def __setitem__(self, k, v): self._shard(k).set(k, v)
    def __getitem__(self, k): return self._shard(k)[k]
    def __delitem__(self, k): del self._shard(k)[k]
    def __contains__(self, key): return key in self._shard(key)
    def __len__(self): return sum(len(s) for s in self._shards)
    def __iter__(self):
        for s in self._shards: yield from s

    def expire(self): return sum(s.expire() for s in self._shards)
    def evict(self, tag): return sum(s.evict(tag) for s in self._shards)
    def clear(self): return sum(s.clear() for s in self._shards)
    def volume(self): return sum(s.volume() for s in self._shards)
    def stats(self, reset=False):
        parts = [s.stats(reset=reset) for s in self._shards]
        return AttrDict(hits=sum(p.hits for p in parts), misses=sum(p.misses for p in parts))
    def close(self):
        for s in self._shards: s.close()
    def __enter__(self): return self
    def __exit__(self, *_): self.close()

    def memoize(self, expire=None, typed=False, tag=None):
        'Decorator: route memoized results to the correct shard.'
        def decorator(func):
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                k = _make_cache_key(func, args, kwargs, typed)
                shard = self._shards[hash(k) % self.shards]
                result = shard.get(k, _MISSING)
                if result is not _MISSING: return result
                with _key_lock(k):
                    result = shard.get(k, _MISSING)
                    if result is not _MISSING: return result
                    result = func(*args, **kwargs)
                    shard.set(k, result, expire=expire, tag=tag)
                return result
            return wrapper
        return decorator

    def memoize_stampede(self, expire=None, typed=False, tag=None, beta=1):
        'Decorator: XFetch stampede protection routed across shards.'
        def decorator(func):
            wrapped = [s.memoize_stampede(expire=expire, typed=typed, tag=tag, beta=beta)(func)
                       for s in self._shards]
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                k = _make_cache_key(func, args, kwargs, typed)
                return wrapped[hash(k) % self.shards](*args, **kwargs)
            return wrapper
        return decorator


In [ ]:
show_doc(FanoutCache)

In [ ]:
fc = FanoutCache(':memory:', shards=8)
for i in range(100): fc.set(f'key:{i}', i * 2)
assert len(fc) == 100
for i in range(100): assert fc.get(f'key:{i}') == i * 2


In [ ]:
fc = FanoutCache(':memory:', shards=4)
fc.set('a', 1); fc.set('b', 2)
assert fc.volume() > 0
assert fc.stats().misses == 0


In [ ]:
fc = FanoutCache(':memory:', shards=4)
calls = 0
@fc.memoize(expire=10)
def fib(n): global calls; calls += 1; return n if n < 2 else fib(n-1) + fib(n-2)
fib(5)
prev = calls
fib(5)
assert calls == prev, 'FanoutCache memoize should not recompute'


In [ ]:
#| eval: false
fc = FanoutCache(':memory:', shards=2, cache_cls=SemanticCache)
fc.set('hello world', 'greeting')
assert fc.get('hello world') == 'greeting'


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()